# Inference

**Serving machine-learning models in production: cluster management, cost optimization, data management, dependency setup, monitoring & observability, performance tuning, resource allocation & scaling, security & compliance, and service integration.**

Inference is the *serving* half of the ML lifecycle: taking a trained model and turning it into a low-latency, high-throughput, cost-controlled service that other systems can call. Unlike training (a batch job that runs to completion), inference is a long-lived online service judged on tail latency, availability, throughput-per-dollar, and correctness under real traffic.

## Table of Contents

1. [Introduction](#introduction)
2. [Key Features](#key-features)
3. [Architecture Overview](#architecture)
4. [Installation](#installation)
5. [Basic Usage](#basic-usage)
6. [Advanced Features](#advanced-features)
7. [Use Cases](#use-cases)
8. [Best Practices](#best-practices)
9. [Common Pitfalls](#pitfalls)
10. [Performance Optimization](#performance)
11. [Production Deployment](#deployment)
12. [Monitoring and Observability](#monitoring)
13. [Troubleshooting](#troubleshooting)
14. [Comparison with Alternatives](#comparison)
15. [Resources](#resources)

## Introduction

### What is it?

**Inference** is the process of running a trained model on new inputs to produce predictions. In an MLOps context, "inference" usually means the *serving system* around the model: the network endpoint, the runtime that executes the forward pass, the autoscaler, the batching layer, and the observability stack. The model itself is a small piece; most operational complexity lives in the serving infrastructure.

There are two dominant patterns:

- **Online (real-time) inference** — synchronous request/response, e.g. a REST/gRPC endpoint behind a load balancer. Optimized for *latency* (p50/p95/p99) and availability.
- **Batch (offline) inference** — score a large dataset on a schedule, e.g. nightly recommendations. Optimized for *throughput* and cost, latency is irrelevant.

A third hybrid, **streaming inference**, consumes from a queue (Kafka, Kinesis) and is judged on end-to-end lag.

### Why use it?

- **Separation of concerns** — decoupling serving from training lets each scale and deploy independently. You retrain weekly but deploy hotfixes to the server hourly.
- **Throughput-per-dollar** — dedicated serving runtimes (vLLM, TensorRT-LLM, ONNX Runtime, Triton) apply batching, quantization, and kernel fusion that a naive `model.predict()` loop cannot, often a 5–20x cost reduction.
- **Elastic scaling** — autoscale to zero off-peak and burst to hundreds of replicas under load, paying only for what you use.
- **Operational guardrails** — canary rollouts, A/B traffic splitting, request validation, rate limiting, and audit logging that a research script lacks.

### When to use it?

- You have a model that must answer **live user or service requests** under an SLO.
- You need to **share one model across many callers** without each embedding the weights.
- You must **scale independently** of the rest of the application, or scale **to zero**.
- You need **versioned, auditable, rollback-able** model deployments rather than ad-hoc scripts.

## Key Features

### Core Capabilities of an Inference Serving Stack

| Feature | Description | Benefit |
|---------|-------------|---------|
| Dynamic / continuous batching | Coalesce concurrent requests into one GPU forward pass; for LLMs, schedule at the token level (continuous batching) | Multiplies GPU throughput with little added latency |
| Autoscaling (incl. scale-to-zero) | HPA/KEDA on QPS, queue depth, or GPU utilization; KServe/Knative can scale to zero | Match capacity to demand, eliminate idle cost |
| Multi-framework runtimes | Triton, ONNX Runtime, TorchServe, vLLM, TF Serving behind one API | Serve PyTorch/TF/ONNX/XGBoost from one platform |
| Model & weight optimization | Quantization (INT8/FP8/INT4), KV-cache paging, kernel fusion, speculative decoding | Lower latency, smaller memory footprint, more req/GPU |
| Versioning & safe rollout | Canary, blue-green, shadow traffic, instant rollback | Deploy without downtime or blast radius |
| Observability hooks | Latency/throughput/error metrics, GPU telemetry, request tracing, data-drift capture | Detect regressions and silent model failures early |
| Multi-model serving | Pack many small models per GPU, load on demand | Higher utilization, lower cost per model |

## Architecture Overview

Understanding the request path of a production inference service:

```
                        +---------------------------+
 client --> Ingress --> | API Gateway / Load Balancer|
  (HTTP/gRPC)           |  - authN/Z, rate limiting  |
                        |  - request validation      |
                        +-------------+--------------+
                                      |
                          (route by model + version)
                                      v
          +-------------------------------------------------+
          |              Serving layer (replicas)           |
          |  +-----------+   +--------------------------+   |
          |  | Pre/post  |   |  Inference runtime       |   |
          |  | processing|-->|  (vLLM/Triton/ONNX/TS)   |   |
          |  +-----------+   |  - dynamic batching      |   |
          |                  |  - GPU/accelerator exec  |   |
          |                  +-----------+--------------+   |
          +------------------------------|------------------+
                                         |
            +----------------+   +-------v--------+   +----------------+
            | Model registry |   |  Feature store |   |  Observability  |
            | (S3/MLflow)    |   |  / cache       |   |  (Prom/OTel)    |
            +----------------+   +----------------+   +----------------+
```

### Components

1. **API gateway / ingress** — terminates TLS, authenticates, rate-limits, and routes to the correct model and version. Often an Envoy/NGINX/Istio layer.
2. **Pre/post-processing** — tokenization, image decode/resize, feature lookups, output formatting. Frequently the real latency bottleneck (CPU-bound, not GPU-bound).
3. **Inference runtime** — the engine that runs the forward pass with batching and accelerator kernels (vLLM, TensorRT-LLM, Triton, ONNX Runtime, TorchServe).
4. **Model registry / storage** — versioned artifacts pulled at startup (MLflow, S3/GCS, OCI image). The source of truth for what is deployed.
5. **Autoscaler** — HPA/KEDA/Knative scaling replicas on QPS, queue depth, or GPU utilization.
6. **Observability** — metrics, traces, logs, and prediction capture for drift and quality monitoring.

## Installation

### Prerequisites

- Python 3.9+
- An accelerator for non-trivial models: NVIDIA GPU + CUDA 12.x drivers (or CPU/ONNX for small models)
- Docker, and a Kubernetes cluster with the NVIDIA device plugin for production
- A model artifact and (optionally) a model registry such as MLflow

### Installation Steps

Pick a runtime that matches the model. The cell below installs common options; **uncomment in Colab** or run locally in a virtualenv.

In [ ]:
# Uncomment the line(s) you need.

# General-purpose, multi-framework server (PyTorch/TF/ONNX/XGBoost):
# !pip install "tritonclient[all]"           # client; server runs as a container

# High-throughput LLM serving (OpenAI-compatible API):
# !pip install vllm

# Lightweight CPU/GPU inference for ONNX models:
# !pip install onnxruntime-gpu        # or onnxruntime for CPU

# Kubernetes-native serving CRDs (installed into the cluster, not pip):
# kubectl apply -f https://github.com/kserve/kserve/releases/download/v0.13.0/kserve.yaml

## Basic Usage

### Quick Start Example

The simplest possible online endpoint: a FastAPI app that loads a model once at startup and serves predictions. This pattern is fine for small CPU models and is the conceptual baseline that dedicated runtimes optimize on.

In [ ]:
# minimal_server.py — a baseline real-time inference endpoint
from fastapi import FastAPI
from pydantic import BaseModel
import numpy as np

app = FastAPI(title="inference-demo")

# Load the model ONCE at process start, not per-request.
# (Swap this stub for joblib.load(...) / torch.load(...) / ort.InferenceSession(...).)
class DummyModel:
    def predict(self, x: np.ndarray) -> np.ndarray:
        return 1.0 / (1.0 + np.exp(-x.sum(axis=1)))  # toy logistic score

model = DummyModel()

class Request(BaseModel):
    features: list[list[float]]

@app.get("/healthz")          # liveness/readiness probe target
def healthz():
    return {"status": "ok"}

@app.post("/predict")
def predict(req: Request):
    x = np.asarray(req.features, dtype=np.float32)
    scores = model.predict(x)
    return {"scores": scores.tolist()}

# Run with:  uvicorn minimal_server:app --host 0.0.0.0 --port 8080 --workers 4
print("Defined FastAPI app with /healthz and /predict")

Calling it from a client looks like any HTTP service. The same shape applies whether the backend is FastAPI, Triton, or vLLM's OpenAI-compatible API — only the URL and payload schema change.

In [ ]:
import json

# Example request payload and an offline simulation of the server's response.
payload = {"features": [[0.5, -1.2, 3.0], [-0.1, 0.0, 0.2]]}

import numpy as np
x = np.asarray(payload["features"], dtype=np.float32)
scores = (1.0 / (1.0 + np.exp(-x.sum(axis=1)))).tolist()
print("POST /predict ->", json.dumps({"scores": scores}))

# With a live server you would instead do:
# import requests
# r = requests.post('http://localhost:8080/predict', json=payload, timeout=2)
# print(r.json())

## Advanced Features

### Dynamic batching, continuous batching, and quantization

#### Dynamic batching

GPUs are throughput devices: one request often wastes >90% of the silicon. **Dynamic batching** holds incoming requests for a few milliseconds and runs them as one batch. The trade-off is a small added queue latency for a large throughput gain. Triton exposes this via `dynamic_batching { max_queue_delay_microseconds: 1000 }` in `config.pbtxt`.

#### Continuous batching (LLMs)

For autoregressive LLMs, request lengths vary wildly, so fixed batches stall on the longest sequence. **Continuous (in-flight) batching** schedules at the *token* level — finished sequences leave the batch and new ones join every step. Combined with **paged KV-cache** (PagedAttention), this is why vLLM and TensorRT-LLM achieve many times the throughput of naive generation.

#### Quantization

Reduce weight/activation precision (FP16 -> INT8 / FP8 / INT4) to shrink memory and speed up matmuls. Post-training quantization (PTQ) is cheap; quantization-aware training (QAT) recovers more accuracy. Always measure quality after quantizing.

In [ ]:
# High-throughput LLM serving with vLLM's continuous batching + paged KV-cache.
# (Requires a GPU and `pip install vllm`; shown here as a reference snippet.)
#
# from vllm import LLM, SamplingParams
#
# llm = LLM(
#     model="meta-llama/Llama-3.1-8B-Instruct",
#     quantization="fp8",        # FP8 weights -> ~half the VRAM, higher throughput
#     gpu_memory_utilization=0.90,
#     max_model_len=8192,
#     enable_chunked_prefill=True,
# )
# params = SamplingParams(temperature=0.7, max_tokens=256)
#
# # vLLM batches these prompts continuously across decode steps:
# prompts = ["Summarize MLOps inference.", "Explain dynamic batching."]
# for out in llm.generate(prompts, params):
#     print(out.outputs[0].text)
#
# Or serve an OpenAI-compatible endpoint with:
#   vllm serve meta-llama/Llama-3.1-8B-Instruct --quantization fp8 --port 8000
print("vLLM reference: continuous batching + FP8 quantization + paged KV-cache")

## Use Cases

### Real-world Applications

#### Use Case 1: Real-time fraud scoring

- **Context** — a payments platform must score each transaction in <50 ms p99 or block checkout.
- **Implementation** — a gradient-boosted model exported to ONNX, served by ONNX Runtime behind gRPC with feature lookups from a low-latency feature store (Redis). HPA on QPS, three availability zones.
- **Results** — single-digit-millisecond inference, with the feature lookup as the dominant latency term; capacity follows daily traffic via autoscaling.

#### Use Case 2: LLM-backed customer support assistant

- **Context** — a chat assistant serving thousands of concurrent sessions with streaming token output.
- **Implementation** — vLLM with continuous batching and FP8 quantization on A100/H100 nodes, KServe for autoscaling, prompt/response logging for evaluation and drift.
- **Results** — high tokens/sec per GPU and predictable cost-per-1k-tokens; scale-to-low replica counts overnight.

#### Use Case 3: Nightly batch recommendations

- **Context** — recompute recommendations for 50M users every night.
- **Implementation** — a Spark/Ray batch job loads the model on each worker and scores partitions; spot/preemptible GPU instances keep cost low; latency is irrelevant.
- **Results** — throughput-optimized, fault-tolerant, and roughly an order of magnitude cheaper than scoring the same volume online.

## Best Practices

1. **Define an SLO first** — pick explicit p50/p95/p99 latency, availability, and cost-per-1k-requests targets. Every architecture choice is a trade-off against these numbers.
2. **Load the model once** — initialize weights at process/worker startup, never per request. Use readiness probes so traffic only arrives after the model is loaded.
3. **Separate readiness from liveness** — liveness restarts a hung process; readiness gates traffic during model load or warmup. Conflating them causes restart storms.
4. **Right-size the runtime** — small models: ONNX Runtime/CPU; large/throughput-critical: Triton/vLLM/TensorRT-LLM on GPU. Don't put a 30B LLM behind a naive Flask loop.
5. **Batch and quantize before buying more GPUs** — dynamic/continuous batching and INT8/FP8 usually beat horizontal scaling on cost-per-request; validate accuracy after quantizing.
6. **Pin and reproduce dependencies** — bake exact CUDA, framework, and model versions into the image; serve the same artifact you tested. Use immutable, versioned image tags.
7. **Roll out safely** — canary or shadow new model versions, watch quality + latency metrics, and keep instant rollback. Treat a model change like a code change.
8. **Warm up before serving** — run a few dummy inferences at startup to trigger CUDA graph capture / JIT compilation so the first real request isn't a multi-second outlier.

## Common Pitfalls

1. **Loading the model per request** — re-reading weights from disk on every call destroys latency and memory. *Avoid:* load once at startup; gate traffic on a readiness probe.
2. **Ignoring tail latency** — averages hide pain; users feel p99. Cold starts, GC pauses, and unbatched stragglers live in the tail. *Avoid:* track p95/p99, warm up, and cap batch wait time.
3. **CPU-bound pre/post-processing on the GPU path** — tokenization, image decode, and feature lookups can dominate a 'GPU' service. *Avoid:* profile end-to-end; move or parallelize pre/post-processing, cache features.
4. **OOM under load** — KV-cache and large batches blow past VRAM only at peak concurrency. *Avoid:* cap `max_model_len`/batch size, set `gpu_memory_utilization`, load-test to the limit.
5. **Silent model drift** — the service is green while prediction quality rots as input distributions shift. *Avoid:* log inputs/outputs, monitor data drift and business KPIs, not just HTTP 200s.
6. **Training/serving skew** — different preprocessing in training vs serving yields subtly wrong predictions. *Avoid:* share one preprocessing library across both paths and test parity.

## Performance Optimization

### Configuration Tuning

Key parameters to tune (names vary by runtime):

- **Batch size / max queue delay** — larger batches raise throughput but add latency; tune `max_batch_size` and `max_queue_delay_microseconds` against your p99 SLO.
- **Precision / quantization** — FP16 vs FP8 vs INT8 vs INT4 trades accuracy for speed and memory; pick the lowest precision that holds quality.
- **Concurrency / instances per GPU** — Triton `instance_group` count and server worker count control how many requests execute in parallel before queuing.
- **KV-cache / `gpu_memory_utilization`** — for LLMs, more cache means longer contexts and bigger batches up to the VRAM ceiling.
- **Hardware kernels** — TensorRT/TensorRT-LLM engine builds, CUDA graphs, and flash-attention cut per-call overhead.

The cell below is a minimal latency/throughput micro-benchmark harness — the first thing to run before and after any tuning change.

In [ ]:
import time
import numpy as np

def benchmark(predict_fn, batch, n_iters=200, warmup=20):
    """Measure throughput and tail latency of a predict function."""
    for _ in range(warmup):          # warm up caches / JIT / CUDA graphs
        predict_fn(batch)
    latencies = []
    for _ in range(n_iters):
        t0 = time.perf_counter()
        predict_fn(batch)
        latencies.append((time.perf_counter() - t0) * 1000.0)  # ms
    latencies = np.array(latencies)
    bs = len(batch)
    return {
        "p50_ms": round(float(np.percentile(latencies, 50)), 3),
        "p95_ms": round(float(np.percentile(latencies, 95)), 3),
        "p99_ms": round(float(np.percentile(latencies, 99)), 3),
        "throughput_rps": round(bs / (latencies.mean() / 1000.0), 1),
    }

# Demo against the toy model from Basic Usage.
def predict_fn(batch):
    x = np.asarray(batch, dtype=np.float32)
    return 1.0 / (1.0 + np.exp(-x.sum(axis=1)))

batch = np.random.randn(32, 3).tolist()
print(benchmark(predict_fn, batch))

## Production Deployment

### Deploying inference in production

#### Docker Deployment

Bake the runtime, dependencies, and a startup that loads the model once. Run an in-image healthcheck and never embed secrets.

```dockerfile
FROM nvcr.io/nvidia/tritonserver:24.05-py3   # or python:3.11-slim for CPU/ONNX

WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Model artifacts are pulled at runtime from the registry, not baked in,
# so the image stays small and one image serves many model versions.
COPY serve.py .

ENV MODEL_URI=s3://models/fraud/v7 \
    OMP_NUM_THREADS=4

EXPOSE 8080
HEALTHCHECK --interval=15s --timeout=2s --retries=3 \
  CMD curl -fsS http://localhost:8080/healthz || exit 1

CMD ["uvicorn", "serve:app", "--host", "0.0.0.0", "--port", "8080", "--workers", "4"]
```

#### Kubernetes Deployment

Request the GPU explicitly, set resource limits, and wire liveness/readiness probes. The Service fronts the autoscaled replicas.

```yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: fraud-inference
spec:
  replicas: 2
  selector:
    matchLabels: { app: fraud-inference }
  template:
    metadata:
      labels: { app: fraud-inference }
    spec:
      containers:
        - name: server
          image: registry.example.com/fraud-inference:v7
          ports: [{ containerPort: 8080 }]
          resources:
            limits:
              nvidia.com/gpu: 1
              memory: 16Gi
            requests:
              cpu: "2"
              memory: 8Gi
          readinessProbe:        # gate traffic until the model is loaded
            httpGet: { path: /healthz, port: 8080 }
            initialDelaySeconds: 20
            periodSeconds: 5
          livenessProbe:         # restart only a truly hung process
            httpGet: { path: /healthz, port: 8080 }
            initialDelaySeconds: 60
            periodSeconds: 15
---
apiVersion: v1
kind: Service
metadata:
  name: fraud-inference
spec:
  selector: { app: fraud-inference }
  ports: [{ port: 80, targetPort: 8080 }]
---
apiVersion: autoscaling/v2
kind: HorizontalPodAutoscaler
metadata:
  name: fraud-inference
spec:
  scaleTargetRef:
    apiVersion: apps/v1
    kind: Deployment
    name: fraud-inference
  minReplicas: 2
  maxReplicas: 20
  metrics:
    - type: Resource
      resource:
        name: cpu
        target: { type: Utilization, averageUtilization: 65 }
```

For GPU-aware autoscaling on custom metrics (QPS, queue depth, GPU util), use **KEDA** or a serving platform like **KServe**/**Knative**, which also supports scale-to-zero.

## Monitoring and Observability

### Monitoring inference in production

#### Key Metrics to Track

- **Latency p50 / p95 / p99** — per model and version; the primary SLO signal. Watch the tail, not the mean.
- **Throughput (RPS / tokens-per-sec) and queue depth** — saturation indicators that should drive autoscaling.
- **Error rate** — HTTP 5xx, timeouts, and OOM kills, split by cause.
- **GPU utilization, memory, and SM occupancy** — via DCGM/`nvidia-smi`; low utilization at high cost means batching or right-sizing is needed.
- **Cost per 1k requests / per 1M tokens** — the business-facing efficiency metric.
- **Data & prediction drift** — input distribution shift and output/score distribution shift; the early warning for silent quality decay.

#### Logging Best Practices

- **Structure logs as JSON** with request id, model version, latency, and outcome so they are queryable.
- **Use appropriate levels** — INFO for lifecycle, WARN for retries/throttling, ERROR for failed inferences; never log full payloads with PII.
- **Sample prediction capture** — store a sampled stream of (input, output) for offline evaluation and drift detection.
- **Emit OpenTelemetry traces** spanning gateway -> preprocessing -> model -> postprocessing to locate the real bottleneck.

The cell below shows Prometheus-style instrumentation of an inference handler.

In [ ]:
# Instrument an inference handler with Prometheus metrics.
# (pip install prometheus-client)
try:
    from prometheus_client import Counter, Histogram

    REQUESTS = Counter("inference_requests_total", "Total requests", ["model", "status"])
    LATENCY = Histogram(
        "inference_latency_seconds", "Inference latency", ["model"],
        buckets=(0.005, 0.01, 0.025, 0.05, 0.1, 0.25, 0.5, 1.0, 2.5),
    )

    def handle(model_name, predict_fn, batch):
        with LATENCY.labels(model_name).time():
            try:
                out = predict_fn(batch)
                REQUESTS.labels(model_name, "ok").inc()
                return out
            except Exception:
                REQUESTS.labels(model_name, "error").inc()
                raise
    msg = "Defined instrumented handle() exposing latency histogram + request counter"
except ImportError:
    msg = "prometheus-client not installed; `pip install prometheus-client` to run this"
print(msg)

## Troubleshooting

#### Issue 1: High tail latency (p99) despite low average

**Symptoms:** p50 is healthy but p99 spikes; intermittent slow requests.

**Cause:** cold starts after scale-up, unbatched stragglers waiting on a long sequence, GC or CUDA-graph capture on the first call, or noisy-neighbor GPU contention.

**Solution:** add startup warmup, cap `max_queue_delay`, use continuous batching for LLMs, pre-scale ahead of known peaks, and pin GPU resources so pods don't share an accelerator.

#### Issue 2: CUDA out-of-memory under load

**Symptoms:** server crashes or returns 5xx only at peak concurrency; `CUDA out of memory` in logs.

**Cause:** KV-cache or batch memory grows with concurrency and context length beyond VRAM.

**Solution:** lower `gpu_memory_utilization`, cap `max_model_len`/`max_batch_size`, enable quantization to shrink weights, and load-test to find the safe concurrency ceiling.

#### Issue 3: Predictions look wrong in production but right in the notebook

**Symptoms:** offline metrics are good; live quality is poor.

**Cause:** training/serving skew — different preprocessing, feature ordering, or a stale model version deployed.

**Solution:** share one preprocessing library across train and serve, add a parity test on a golden dataset, and assert the deployed model hash matches the registry.

## Comparison with Alternatives

### How common inference runtimes compare

| Feature | Triton Inference Server | vLLM | TorchServe | ONNX Runtime |
|---------|-------------------------|------|------------|--------------|
| Best for | Multi-framework, mixed workloads | High-throughput LLMs | PyTorch models | Lightweight CPU/GPU, portable |
| Frameworks | PyTorch, TF, ONNX, TensorRT, Python | Transformer LLMs | PyTorch (+ ONNX via handler) | ONNX (export from any) |
| Batching | Dynamic + sequence batching | Continuous (token-level) + paged KV | Dynamic batching | App-managed |
| LLM features | Via TensorRT-LLM backend | First-class (PagedAttention, speculative) | Limited | Limited |
| Footprint | Heavy, feature-rich | GPU-focused | Moderate | Very light |

### When to Choose This Approach

Choose a **dedicated inference runtime** (over a hand-rolled Flask/FastAPI loop) when:

- You serve on **GPUs** and need batching to hit acceptable cost-per-request.
- You serve **LLMs** and need continuous batching, paged KV-cache, or quantization.
- You must serve **many models/frameworks** behind one consistent, observable API.

Stick with a **simple FastAPI/ONNX-Runtime service** when the model is small, CPU-bound, low-QPS, and the operational simplicity outweighs raw throughput.

## Resources

### Official Documentation

- NVIDIA Triton Inference Server: https://github.com/triton-inference-server/server
- vLLM: https://docs.vllm.ai/
- KServe: https://kserve.github.io/website/
- TorchServe: https://pytorch.org/serve/
- ONNX Runtime: https://onnxruntime.ai/docs/
- TensorRT-LLM: https://github.com/NVIDIA/TensorRT-LLM

### Tutorials and Guides

- Triton model configuration & dynamic batching: https://github.com/triton-inference-server/server/blob/main/docs/user_guide/model_configuration.md
- vLLM PagedAttention / continuous batching paper: https://arxiv.org/abs/2309.06180
- Knative/KServe autoscaling guide: https://kserve.github.io/website/latest/modelserving/autoscaling/autoscaling/

### Community Resources

- vLLM Discord & GitHub Discussions: https://github.com/vllm-project/vllm/discussions
- Triton GitHub Discussions: https://github.com/triton-inference-server/server/discussions
- Stack Overflow tags: `nvidia-triton`, `vllm`, `onnxruntime`

### Related Technologies

- **Model registries:** MLflow, BentoML, Hugging Face Hub
- **Serving platforms:** KServe, Seldon Core, Ray Serve, BentoML
- **Optimization & compilers:** TensorRT, ONNX, OpenVINO, torch.compile